# Post-fit pulls in physical parameters — Asimov fit results

This compact notebook transforms each fit's **joint MCMC chain** from standardized fit coordinates to the physical $M_A$ or complete $z$-expansion coefficient vector. For correlated priors it applies the inverse PCA map. The MINERvA kmax=6 uniform fit uses the same PCA spline directions with flat penalties; its corner plot is posterior-only because no informative prior enters the fit. Dependent coefficients preserve $F_A(0)$ and the four sum rules.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().resolve()
while repo.name != 'axial_mass' and repo != repo.parent:
    repo = repo.parent
helper_dir = repo / 'ma_zexp' / 'python' / 'scripts'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import (
    FIGURE_ROOT, plot_ma_posterior_overlay, run_suite,
)

## Run every measurement

Set `BURN_IN` or `THIN` if needed. Each measurement produces one table and one two-row figure: transformed marginal distributions above, prior/post-fit intervals below.

In [ ]:
# Optional: fix the display range for parameters shared across fits (e.g. all
# M_A fits, or a1/a2 across every z-expansion prior) so their marginal and
# corner panels are directly comparable. Keys are parameter names as they
# appear in a fit's results table (e.g. 'M_A [GeV]', 'a1', 'a2', ...);
# parameters not listed here keep their automatic, per-fit range. Leave this
# dict empty to keep every parameter's range automatic, as before.

# Names:
# M_A [GeV]
# AxFFCCQEshape pull 
# NormCCMEC pull
# RPA CCQE pull

AXIS_RANGES = {
    'M_A [GeV]': (0.8, 1.8),
    'NormCCMEC pull': (-2, 3),
    'RPA CCQE pull': (-3, 3),
}

In [ ]:
BURN_IN = 0
THIN = 1
N_PRIOR_SAMPLES = 50_000
SHOW_PRIOR_IN_CORNER = True
SAVE_FIGURES = True
SAVE_DPI = 600

results = run_suite(
    'asimov_fit_results', BURN_IN, THIN, N_PRIOR_SAMPLES,
    show_prior_in_corner=SHOW_PRIOR_IN_CORNER,
    save_figures=SAVE_FIGURES, save_dpi=SAVE_DPI,
    axis_ranges=AXIS_RANGES,
)

## $M_A$ fit with and without its pull penalty

This corner overlays the joint posteriors for physical $M_A$, `NormCCMEC`, and `RPA_CCQE`. Both fits omit `AxFFCCQEshape`; the only prior difference is that `ma_uniform` removes the Gaussian pull penalty from $M_A$.

In [ ]:
MA_OVERLAY_FITS = ('ma_no_axff', 'ma_uniform')
MA_OVERLAY_LABELS = {
    'ma_no_axff': r'Posterior from Gaussian $M_A$ prior',
    'ma_uniform': r'Posterior from uniform $M_A$ prior',
}

ma_overlay_corner = plot_ma_posterior_overlay(
    results, MA_OVERLAY_FITS, labels=MA_OVERLAY_LABELS,
    figsize=(7.2, 7.2),
)
ma_overlay_dir = FIGURE_ROOT / 'asimov_fit_results' / 'comparison_overlays'
ma_overlay_dir.mkdir(parents=True, exist_ok=True)
for extension in ('pdf',):
    ma_overlay_corner.savefig(
        ma_overlay_dir / f'ma_overlay.{extension}', dpi=600,
        bbox_inches='tight', pad_inches=.03, facecolor='white',
    )
ma_overlay_corner